#**Text Classification - Sentiment Analysis in GCP**

In [ ]:
if 'google.colab' in str(get_ipython()):
  print('Running on CoLab')
  RunningInColab = True
else:
  print('Not running on CoLab')
  RunningInColab = False

Running on CoLab


In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sqrt
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, DoubleType
from scipy.stats import pearsonr
from itertools import combinations

project_id = 'ymehta9-cis415-2025springc'

bucket = 'ymehta9_data_for_gcp_labs'

path_to_data_files = "/data_for_assignment/"
amazon_reviews_file_name = "data_Amazon_Reviews.csv"
relative_path_to_file = path_to_data_files[1:] + amazon_reviews_file_name
full_file_path = "gs://" + bucket + "/" + relative_path_to_file

print(f"ProjectID (and not the Project Name) is: {project_id}")
print(f"Bucket name is: {bucket}")

ProjectID (and not the Project Name) is: ymehta9-cis415-2025springc
Bucket name is: ymehta9_data_for_gcp_labs


In [ ]:
from google.cloud import storage
client = storage.Client()
print(f"Package imports done")

if RunningInColab == True:

  from google.cloud import storage
  import google.auth

  !pip install gcsfs
  import gcsfs

  from google.colab import auth
  auth.authenticate_user()

  credentials, default_project_id = google.auth.default()
  !gcloud config set project {project_id}

Package imports done
Updated property [core/project].


In [ ]:
fs = gcsfs.GCSFileSystem(project=project_id)

fs.ls('ymehta9_data_for_gcp_labs/data_for_assignment/')


['ymehta9_data_for_gcp_labs/data_for_assignment/',
 'ymehta9_data_for_gcp_labs/data_for_assignment/data_Amazon_Reviews.csv',
 'ymehta9_data_for_gcp_labs/data_for_assignment/missionImpossible_rating.csv',
 'ymehta9_data_for_gcp_labs/data_for_assignment/movie_ratings.csv']

In [ ]:
if RunningInColab:
  spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()


In [6]:
print("Driver Memory:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("Executor Memory:", spark.sparkContext.getConf().get("spark.executor.memory"))

Driver Memory: 8g
Executor Memory: 8g


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import Tokenizer, StopWordsRemover, Word2Vec
from pyspark.sql.functions import col, regexp_replace
from google.cloud import storage

In [8]:
!pip install nltk

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stops = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **Preparing train and test data**

In [ ]:
temp_file_path = "/content/" + amazon_reviews_file_name if RunningInColab else full_file_path

# Reload CSV from GCS
storage_client = storage.Client()
bucket_object = storage_client.bucket(bucket)
blob = bucket_object.blob(relative_path_to_file)
blob.download_to_filename(temp_file_path)

# Read CSV again
spark_df = spark.read.csv(temp_file_path, sep=",", header=True)

# Rename and clean
spark_df = spark_df.withColumnRenamed("Review ", "Review")
spark_df = spark_df.withColumn("label", col("Polarity").cast(DoubleType()))
spark_df = spark_df.na.drop(subset=["Review", "label"])
spark_df.show(5)


+--------+--------------------+--------------------+-----+
|Polarity|               Title|              Review|label|
+--------+--------------------+--------------------+-----+
|       2|            Great CD|"My lovely Pat ha...|  2.0|
|       2|One of the best g...|Despite the fact ...|  2.0|
|       1|Batteries died wi...|I bought this cha...|  1.0|
|       2|works fine, but M...|Check out Maha En...|  2.0|
|       2|Great for the non...|Reviewed quite a ...|  2.0|
+--------+--------------------+--------------------+-----+
only showing top 5 rows



In [12]:
spark_df.columns

['Polarity', 'Title', 'Review', 'label']

In [13]:
print(locals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'RunningInColab', '_i2', 'SparkSession', 'col', 'avg', 'sqrt', 'StructType', 'StructField', 'FloatType', 'IntegerType', 'DoubleType', 'pearsonr', 'combinations', 'project_id', 'bucket', 'path_to_data_files', 'amazon_reviews_file_name', 'relative_path_to_file', 'full_file_path', '_i3', 'storage', 'client', 'google', '_exit_code', 'gcsfs', 'auth', 'credentials', 'default_project_id', '_i4', 'fs', '_4', '_i5', 'spark', '_i6', '_i7', 'Pipeline', 'RandomForestClassifier', 'MulticlassClassificationEvaluator', 'Tokenizer', 'StopWordsRemover', 'Word2Vec', 'regexp_replace', '_i8', '_i9', 'nltk', 'stopwords', 'stops', '_i10', '_i11', 'temp_file_path', 'storage_client', 'bucket_object', 'blob', 'spark_df', '_i12', '_12', '_i13'])


In [ ]:
if RunningInColab == True:

  storage_client = storage.Client()

  temp_file_path = "/content/"+ amazon_reviews_file_name

  print(f"bucket_name={bucket}")
  print(f"relative_path_to_file = {relative_path_to_file}")
  print(full_file_path)
  print(f"Downloading the file from GCS to a local file in Colab")
  bucket_object = storage_client.bucket(bucket)
  blob = bucket_object.blob(relative_path_to_file)
  blob.download_to_filename(temp_file_path)
  print(f"Reading the local file using spark")

  spark_df = spark.read.csv(temp_file_path, sep=",", header=True)
else:
  print(f"Reading the file using spark directly from GCS")
temp_file_path = "/content/" + amazon_reviews_file_name if RunningInColab else full_file_path


spark_df = spark_df.withColumn("label", col("Polarity").cast(DoubleType()))

spark_df.show()

print(f"Total number of records from data file = {spark_df.count()}")



bucket_name=ymehta9_data_for_gcp_labs
relative_path_to_file = data_for_assignment/data_Amazon_Reviews.csv
gs://ymehta9_data_for_gcp_labs/data_for_assignment/data_Amazon_Reviews.csv
Reading the local file using spark
+--------+--------------------+--------------------+-----+
|Polarity|               Title|             Review |label|
+--------+--------------------+--------------------+-----+
|       2|            Great CD|"My lovely Pat ha...|  2.0|
|       2|One of the best g...|Despite the fact ...|  2.0|
|       1|Batteries died wi...|I bought this cha...|  1.0|
|       2|works fine, but M...|Check out Maha En...|  2.0|
|       2|Great for the non...|Reviewed quite a ...|  2.0|
|       1|DVD Player crappe...|I also began havi...|  1.0|
|       1|      Incorrect Disc|I love the style ...|  1.0|
|       1|DVD menu select p...|I cannot scroll t...|  1.0|
|       2|Unique Weird Orie...|"Exotic tales of ...|  2.0|
|       1|"Not an ""ultimat...|Firstly,I enjoyed...|  1.0|
|       2|Great b

In [17]:
spark_df = spark_df.withColumn("label", (col("Polarity") - 1).cast(DoubleType()))

In [18]:
spark_df.groupBy("label").count().show()

+-----+------+
|label| count|
+-----+------+
|  0.0|200000|
|  1.0|200000|
+-----+------+



In [ ]:
import string, re
from pyspark.sql.functions import regexp_replace, col

spark_df = spark_df.withColumnRenamed("Review ", "review")
spark_df = spark_df.withColumn("label", col("Polarity").cast(DoubleType()))

train, test = spark_df.randomSplit([0.8, 0.2], seed=42)
punct_pattern = "[" + re.escape(string.punctuation) + "]"

train = train.withColumn("noPunc_texts", regexp_replace(col("review"), punct_pattern, " "))
test = test.withColumn("noPunc_texts", regexp_replace(col("review"), punct_pattern, " "))

In [ ]:
import string
print(string.punctuation)


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [ ]:
from pyspark.sql.functions import regexp_replace

punct_pattern = "[" + re.escape(string.punctuation) + "]"

train_no_punc = train.withColumn("text", regexp_replace("review", punct_pattern, " "))
test_no_punc = test.withColumn("text", regexp_replace("review", punct_pattern, " "))

train_with_punc = train.withColumn("text", col("review"))
test_with_punc = test.withColumn("text", col("review"))

In [26]:
train_no_punc.show()

+--------+--------------------+--------------------+-----+--------------------+--------------------+
|Polarity|               Title|              review|label|        noPunc_texts|                text|
+--------+--------------------+--------------------+-----+--------------------+--------------------+
|       1|                NULL|"Some good rockin...|  1.0| Some good rockin...| Some good rockin...|
|       1|                NULL|I can't give you ...|  1.0|I can t give you ...|I can t give you ...|
|       1| !!! NOT 2 BOGUS !!!|THESE GUYS ARE TI...|  1.0|THESE GUYS ARE TI...|THESE GUYS ARE TI...|
|       1|!!!! DO NOT PURCH...|This thermometer ...|  1.0|This thermometer ...|This thermometer ...|
|       1|!!!Not enough to ...|Make sure you che...|  1.0|Make sure you che...|Make sure you che...|
|       1|!!DO NOT GET THIS...|The G key on arri...|  1.0|The G key on arri...|The G key on arri...|
|       1|           !WARNING!|This might seem l...|  1.0|This might seem l...|This might s

In [27]:
test_no_punc.show()

+--------+--------------------+--------------------+-----+--------------------+--------------------+
|Polarity|               Title|              review|label|        noPunc_texts|                text|
+--------+--------------------+--------------------+-----+--------------------+--------------------+
|       1|                   !|This doll is VERY...|  1.0|This doll is VERY...|This doll is VERY...|
|       1|   !!!PLEASE NOTE!!!|Please read this,...|  1.0|Please read this ...|Please read this ...|
|       1|!!Why pay for thi...|If you want to ba...|  1.0|If you want to ba...|If you want to ba...|
|       1|""" Great Fiction...|"I have read 11 b...|  1.0| I have read 11 b...| I have read 11 b...|
|       1|       """0"" stars"|"No matter how yo...|  1.0| No matter how yo...| No matter how yo...|
|       1|"""A Very Weak Ef...|It is apparent th...|  1.0|It is apparent th...|It is apparent th...|
|       1|"""Along the stre...|This book came hi...|  1.0|This book came hi...|This book ca

In [43]:
train_with_punc.show()

+--------+--------------------+--------------------+-----+--------------------+--------------------+
|Polarity|               Title|              review|label|        noPunc_texts|                text|
+--------+--------------------+--------------------+-----+--------------------+--------------------+
|       1|                NULL|"Some good rockin...|  1.0| Some good rockin...|"Some good rockin...|
|       1|                NULL|I can't give you ...|  1.0|I can t give you ...|I can't give you ...|
|       1| !!! NOT 2 BOGUS !!!|THESE GUYS ARE TI...|  1.0|THESE GUYS ARE TI...|THESE GUYS ARE TI...|
|       1|!!!! DO NOT PURCH...|This thermometer ...|  1.0|This thermometer ...|This thermometer ...|
|       1|!!!Not enough to ...|Make sure you che...|  1.0|Make sure you che...|Make sure you che...|
|       1|!!DO NOT GET THIS...|The G key on arri...|  1.0|The G key on arri...|The G key on arri...|
|       1|           !WARNING!|This might seem l...|  1.0|This might seem l...|This might s

In [60]:
test_with_punc.show()

+--------+--------------------+--------------------+-----+--------------------+--------------------+
|Polarity|               Title|              review|label|        noPunc_texts|                text|
+--------+--------------------+--------------------+-----+--------------------+--------------------+
|       1|                   !|This doll is VERY...|  1.0|This doll is VERY...|This doll is VERY...|
|       1|   !!!PLEASE NOTE!!!|Please read this,...|  1.0|Please read this ...|Please read this,...|
|       1|!!Why pay for thi...|If you want to ba...|  1.0|If you want to ba...|If you want to ba...|
|       1|""" Great Fiction...|"I have read 11 b...|  1.0| I have read 11 b...|"I have read 11 b...|
|       1|       """0"" stars"|"No matter how yo...|  1.0| No matter how yo...|"No matter how yo...|
|       1|"""A Very Weak Ef...|It is apparent th...|  1.0|It is apparent th...|It is apparent th...|
|       1|"""Along the stre...|This book came hi...|  1.0|This book came hi...|This book ca

# **Tokenization**

In [ ]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
stopwords_cleaner = StopWordsRemover(inputCol="tokens", outputCol="cleaned_text")


# **Stopwords Removal**
<br><br>


In [31]:
stopwords_cleaner = StopWordsRemover(inputCol="tokens", outputCol="cleaned_text", stopWords=stops)

In [ ]:
# Tokenize
train_tok = tokenizer.transform(train_no_punc)
test_tok = tokenizer.transform(test_no_punc)

#Remove stopwords
train_cleaned = stopwords_cleaner.transform(train_tok)
test_cleaned = stopwords_cleaner.transform(test_tok)

# Checking outputs
train_cleaned.select("tokens", "cleaned_text").show(truncate=False)
test_cleaned.select("tokens", "cleaned_text").show(truncate=False)


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------

# **Vectorization: Word2Vec**

In this assignment, we use word2vec vectorization to convert text to numeric representation.

In [33]:
word2Vec = Word2Vec(inputCol="cleaned_text", outputCol="features", vectorSize=100, minCount=1)

**Machine Learning Example - with Random Forest algorithm**

In the following code cells, we use Random Forest classifier algorithm to build the Spark ML pipeline model and train the data.

In [ ]:
from pyspark.ml.feature import Word2Vec
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

word2vec_rf = RandomForestClassifier()

# Vectorizer
word2Vec = Word2Vec(inputCol="cleaned_text", outputCol="features", vectorSize=100, minCount=1)

# Classifier
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20)

# Pipeline
pipeline = Pipeline(stages=[word2Vec, rf])


In [ ]:
# Use preprocessed data (IMPORTANT: not raw train/test)
train_sample = train_cleaned.limit(1000)
test_sample = test_cleaned.limit(500)

# Fit the model
word2vec_rf_model = pipeline.fit(train_sample)

# Make predictions on test data
word2vec_rf_predictions = word2vec_rf_model.transform(test_sample)

# Show predictions
word2vec_rf_predictions.select("label", "prediction", "features").show(truncate=False)


+-----+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [40]:
word2vec_rf_predictions.show(truncate=False)

+--------+--------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------------------------------------

In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="f1")
word2vec_rf_f1_score = evaluator.evaluate(word2vec_rf_predictions)

In [ ]:
print(f"When we remove punctuation marks, f1 score for Word2Vec vectorization in Random Forest algorithm is {word2vec_rf_f1_score}")


When we remove punctuation marks, f1 score for Word2Vec vectorization in Random Forest algorithm is 1.0


#Task

Please provide in this text cell the F1 scores for each model situation and explain which model is the best.

In [ ]:
import os

bucket_name = "ymehta9_data_for_gcp_labs"
bucket = client.get_bucket(bucket_name)
word2vec_rf_model.save("/content/word2vec_rf_model")

#
model_path = "/content/word2vec_rf_model"  # or your local model path
gcs_model_path = "models/word2vec_rf_model"  # path within the GCS bucket

for root, dirs, files in os.walk(model_path):
    for file in files:
        local_file = os.path.join(root, file)
        relative_path = os.path.relpath(local_file, model_path)
        blob = bucket.blob(os.path.join(gcs_model_path, relative_path))
        blob.upload_from_filename(local_file)

print("Model uploaded to GCS.")

Model uploaded to GCS.
